# Claim Consistency + NetworkX Demo — Acc 2 v1

Kiểm tra code bằng dữ liệu giả lập có `claim_id` và timestamp. Đây là engineering demo, không phải metric phát hiện fraud.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","networkx>=3.3,<4"], check=True)
import networkx as nx
from datetime import datetime

cutoff = datetime.fromisoformat("2026-10-11T12:00:00+00:00")
nodes = [
 {"id":"CLM-001","entity_type":"claim","available_at":"2026-10-01T00:00:00+00:00"},
 {"id":"CLM-002","entity_type":"claim","available_at":"2026-10-02T00:00:00+00:00"},
 {"id":"PHONE-01","entity_type":"phone","available_at":"2026-09-01T00:00:00+00:00"},
]
edges = [
 {"source":"CLM-001","target":"PHONE-01","relation":"uses","event_time":"2026-10-01T01:00:00+00:00"},
 {"source":"CLM-002","target":"PHONE-01","relation":"uses","event_time":"2026-10-02T01:00:00+00:00"},
]
graph = nx.Graph()
for node in nodes:
    if datetime.fromisoformat(node["available_at"]) <= cutoff: graph.add_node(node["id"], **node)
for edge in edges:
    if datetime.fromisoformat(edge["event_time"]) <= cutoff: graph.add_edge(edge["source"], edge["target"], **edge)
features = {
 "graph_degree":graph.degree("CLM-001"),
 "shared_entity_count":sum(graph.degree(x)>1 for x in graph.neighbors("CLM-001")),
 "connected_component_size":len(nx.node_connected_component(graph,"CLM-001")),
 "two_hop_claim_count":len({n for x in graph.neighbors("CLM-001") for n in graph.neighbors(x) if n!="CLM-001" and graph.nodes[n]["entity_type"]=="claim"}),
}
assert features == {"graph_degree":1,"shared_entity_count":1,"connected_component_size":3,"two_hop_claim_count":1}
print("Point-in-time graph demo: PASS", features)


In [ ]:
def amount_check(claim_amount, document_amount, confidence, tolerance=0.05):
    if document_amount is None or confidence < 0.5:
        return {"status":"unknown", "reason":"missing_or_low_confidence"}
    ratio = abs(document_amount-claim_amount)/max(abs(claim_amount),1)
    return {"status":"pass" if ratio<=tolerance else "fail", "difference":document_amount-claim_amount}

assert amount_check(10_000_000,10_200_000,0.9)["status"] == "pass"
assert amount_check(10_000_000,12_000_000,0.9)["status"] == "fail"
assert amount_check(10_000_000,None,0.0)["status"] == "unknown"
print("Consistency demo: PASS")


Dữ liệu thật cần claim/entity/edge có timestamp. Không ghép các public dataset theo dòng để tạo graph giả.
